# Part1 : Word translation

In [7]:
import tensorflow as tf
print(tf.__version__)

2.15.0


# Sentence translation - LSTM

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, TimeDistributed, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import GRU


# 1. YOUR DATA
input_sentences = [
    "i am happy",
    "i am sad",
    "he is good",
    "she is good",
    "i like food",
    "i like tea",
    "he likes tea",
    "she likes food"
]

target_sentences = [
    "मैं खुश हूँ",
    "मैं उदास हूँ",
    "वह अच्छा है",
    "वह अच्छी है",
    "मुझे खाना पसंद है",
    "मुझे चाय पसंद है",
    "उसे चाय पसंद है",
    "उसे खाना पसंद है"
]

# 2. Tokenization
input_tokenizer = Tokenizer()
target_tokenizer = Tokenizer()

input_tokenizer.fit_on_texts(input_sentences)
target_tokenizer.fit_on_texts(target_sentences)

input_seq = input_tokenizer.texts_to_sequences(input_sentences)
target_seq = target_tokenizer.texts_to_sequences(target_sentences)

# 3. Padding
max_len = max(
    max(len(seq) for seq in input_seq),
    max(len(seq) for seq in target_seq)
)

input_seq = pad_sequences(input_seq, maxlen=max_len, padding='post')
target_seq = pad_sequences(target_seq, maxlen=max_len, padding='post')

# 4. One-hot encode targets
vocab_size_target = len(target_tokenizer.word_index) + 1

y = np.zeros((len(target_seq), max_len, vocab_size_target))

for i, seq in enumerate(target_seq):
    for t, word in enumerate(seq):
        if word != 0:
            y[i, t, word] = 1

# 5. Model (Embedding + LSTM)
vocab_size_input = len(input_tokenizer.word_index) + 1

model = Sequential()
model.add(Embedding(vocab_size_input, 16, input_length=max_len))

model.add(LSTM(64, return_sequences=True))   # USE THIS FOR LSTM
# model.add(GRU(64, return_sequences=True))    # USE THIS FOR GRU

model.add(TimeDistributed(Dense(vocab_size_target, activation='softmax')))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 6. Train
model.fit(input_seq, y, epochs=200, verbose=0)

# 7. Reverse mapping
reverse_target_index = {v: k for k, v in target_tokenizer.word_index.items()}

# 8. Translation function
def translate(sentence):
    seq = input_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(seq, verbose=0)

    output_words = []
    for t in range(pred.shape[1]):
        index = np.argmax(pred[0, t])
        if index != 0:
            word = reverse_target_index.get(index, '')
            output_words.append(word)

    return " ".join(output_words)





मुझे खुश हूँ हूँ
मुझे चाय पसंद है
उसे अच्छी है है


In [3]:
# 9. TEST

print(translate("i am happy"))
print(translate("i like tea"))
print(translate("he is good"))
print(translate("Good is he?"))

# print(translate("your test sentence here")) # Not perfect

मुझे खुश हूँ हूँ
मुझे चाय पसंद है
उसे अच्छी है है
है है है है
